# SNU AI NetVLAD Token-Fusion Five-Fold Experiment

This CV-only experiment removes ConvNeXt. A frozen ImageNet-1K ResNet-18 produces 7x7 local descriptors, a trainable NetVLAD layer aggregates them into 16 residual cluster tokens, and the existing Qwen token-fusion structured scorer performs permutation scoring. It keeps the MD5 group-stratified five-fold protocol and intentionally contains no final full-data training, test inference, or submission path.

## 1. Paths and experiment parameters

`QWEN_LAYER_INDICES=(-1,)` uses only the final decoder state. The NetVLAD cache is separate from every ConvNeXt cache because both the visual backbone and cache fingerprint change.

In [7]:
import os
from pathlib import Path

# Must be set before the first torch import; reduces CUDA fragmentation in Colab.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

DRIVE_ROOT = Path(os.environ.get('SNU_DRIVE_ROOT', '/content/drive/MyDrive/snu ai'))
BUNDLE_PATH = DRIVE_ROOT / 'snu_colab_full_train_bundle.tar'
CODE_BUNDLE_PATH = DRIVE_ROOT / 'snu_colab_token_fusion_netvlad_code.tar'
WORKDIR = Path(os.environ.get('SNU_WORKDIR', '/content/snu_ai'))
GROUP_FOLDS_CSV = WORKDIR / 'snuaichallenge_data/splits/group_stratified_folds_5.csv'
DRIVE_GROUP_FOLDS_CSV = DRIVE_ROOT / 'splits/group_stratified_folds_5.csv'
LOCAL_TOKEN_CACHE = WORKDIR / 'feature_cache/token_fusion_netvlad_resnet18_v1'
DRIVE_TOKEN_SHARDS = DRIVE_ROOT / 'cache/token_fusion_netvlad_resnet18_v1_shards'
DRIVE_RESULTS_ROOT = DRIVE_ROOT / 'results/token_fusion_netvlad_group_stratified_v1'

QWEN_LAYER_INDICES = (-1,)
FEATURE_BATCH_CANDIDATES = (64, 48, 32, 24, 16, 8)
IMAGE_WORKERS = 16
CACHE_SHARD_SIZE = 100
RESIDENT_STORAGE = 'int8'
HIDDEN_DIM = 256
NUM_HEADS = 4
NETVLAD_CLUSTERS = 16
RESAMPLER_TOKENS = 8
USE_MODALITY_EMBEDDINGS = True
MAX_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TOP_H = 5
TOP_M = 8
ALPHA, BETA, GAMMA, DELTA = 0.1, 1.0, 1.0, 1.0
SEED = 42
FOLDS_TO_RUN = [0, 1, 2, 3, 4]
VERIFIER = 'transformer'


## 2. Mount Drive, install dependencies, and extract code/data

In [8]:
import subprocess
import shutil
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==5.13.0', 'accelerate==1.14.0', 'pillow>=10',
], check=True)
for path in (DRIVE_TOKEN_SHARDS, DRIVE_RESULTS_ROOT, LOCAL_TOKEN_CACHE):
    path.mkdir(parents=True, exist_ok=True)
if not CODE_BUNDLE_PATH.is_file():
    raise FileNotFoundError(CODE_BUNDLE_PATH)
WORKDIR.mkdir(parents=True, exist_ok=True)
subprocess.run(['tar', '-xf', str(CODE_BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
for cache_dir in (WORKDIR / 'snu_ordering').rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)
for name in list(sys.modules):
    if name == 'snu_ordering' or name.startswith('snu_ordering.'):
        del sys.modules[name]
sys.path.insert(0, str(WORKDIR))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Inspect and explicitly confirm the training GPU

Run the first cell, inspect its output, then copy the exact GPU name and rounded VRAM value into the confirmation cell. Feature extraction and training remain blocked until the user confirms them.

In [9]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime.')
DETECTED_GPU_NAME = torch.cuda.get_device_name(0)
DETECTED_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print('GPU:', DETECTED_GPU_NAME)
print(f'VRAM: {DETECTED_VRAM_GB:.2f} GiB')
print('Capability:', torch.cuda.get_device_capability(0))


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 94.97 GiB
Capability: (12, 0)


In [10]:
# Fill these only after checking the preceding cell with the user.
CONFIRMED_GPU_NAME = "NVIDIA RTX PRO 6000 Blackwell Server Edition"
CONFIRMED_VRAM_GB = 94.97

if CONFIRMED_GPU_NAME != DETECTED_GPU_NAME:
    raise RuntimeError(
        f'GPU confirmation mismatch: detected={DETECTED_GPU_NAME!r}, '
        f'confirmed={CONFIRMED_GPU_NAME!r}'
    )
if CONFIRMED_VRAM_GB is None or abs(CONFIRMED_VRAM_GB - DETECTED_VRAM_GB) > 0.5:
    raise RuntimeError(
        f'VRAM confirmation mismatch: detected={DETECTED_VRAM_GB:.2f} GiB, '
        f'confirmed={CONFIRMED_VRAM_GB!r}'
    )
print('GPU configuration confirmed; extraction/training cells are enabled.')


GPU configuration confirmed; extraction/training cells are enabled.


## 4. Build and validate the MD5 group-stratified 24-class folds

In [11]:
from collections import Counter

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(BUNDLE_PATH)
if (not (WORKDIR / 'snuaichallenge_data/train.csv').is_file()
        or not (WORKDIR / 'snuaichallenge_data/train').is_dir()):
    subprocess.run(['tar', '-xf', str(BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
# The large train bundle contains an older code snapshot. Always reapply the
# current code overlay after data extraction, then invalidate cached imports.
subprocess.run(['tar', '-xf', str(CODE_BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
for cache_dir in (WORKDIR / 'snu_ordering').rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)
for name in list(sys.modules):
    if name == 'snu_ordering' or name.startswith('snu_ordering.'):
        del sys.modules[name]
sys.path.insert(0, str(WORKDIR))
from snu_ordering.data import load_samples

import ast
import hashlib
import json
import shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import networkx as nx
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

TRAIN_CSV = WORKDIR / "snuaichallenge_data/train.csv"
IMAGE_ROOT = WORKDIR / "snuaichallenge_data/train"
GROUP_SPLIT_SEED = 42
GROUP_SPLIT_FOLDS = 5
MAX_SAMPLES_PER_HASH = 3
GROUP_SPLIT_METHOD = "md5_stratified_group_kfold_v1"


def _compute_md5(path):
    with path.open("rb") as file:
        return hashlib.md5(file.read()).hexdigest()


def _build_md5_group_stratified_folds(train_csv, image_root):
    train = pd.read_csv(train_csv)
    train["answer_tuple"] = train["Answer"].map(
        lambda value: str(ast.literal_eval(value))
    )

    image_records = []
    for _, row in train.iterrows():
        for image_index in range(1, 5):
            image_records.append(
                {
                    "sample_id": row["Id"],
                    "path": image_root / str(row["Id"]) / str(row[f"Input_{image_index}"]),
                }
            )

    print(f"Computing MD5 for {len(image_records)} training images...")
    with ThreadPoolExecutor(max_workers=32) as executor:
        hashes = list(executor.map(_compute_md5, [record["path"] for record in image_records]))
    for record, image_hash in zip(image_records, hashes, strict=True):
        record["hash"] = image_hash

    hash_counter = defaultdict(set)
    for record in image_records:
        hash_counter[record["hash"]].add(record["sample_id"])
    hash_to_samples = {
        image_hash: sample_ids
        for image_hash, sample_ids in hash_counter.items()
        if len(sample_ids) <= MAX_SAMPLES_PER_HASH
    }

    graph = nx.Graph()
    graph.add_nodes_from(train["Id"].unique())
    for sample_ids in hash_to_samples.values():
        sample_ids = list(sample_ids)
        for left in range(len(sample_ids)):
            for right in range(left + 1, len(sample_ids)):
                graph.add_edge(sample_ids[left], sample_ids[right])

    components = list(nx.connected_components(graph))
    id_to_cluster = {
        sample_id: cluster_id
        for cluster_id, component in enumerate(components)
        for sample_id in component
    }
    train["cluster_id"] = train["Id"].map(id_to_cluster)

    splitter = StratifiedGroupKFold(
        n_splits=GROUP_SPLIT_FOLDS,
        shuffle=True,
        random_state=GROUP_SPLIT_SEED,
    )
    train["Fold"] = -1
    for fold, (_, validation_indices) in enumerate(
        splitter.split(
            train,
            train["answer_tuple"],
            groups=train["cluster_id"],
        )
    ):
        train.loc[validation_indices, "Fold"] = fold

    train["split_method"] = GROUP_SPLIT_METHOD
    train["split_seed"] = GROUP_SPLIT_SEED
    train["split_max_samples_per_hash"] = MAX_SAMPLES_PER_HASH
    train["split_sklearn_version"] = sklearn.__version__
    build_summary = {
        "clusters": len(components),
        "excluded_frequent_hashes": sum(
            len(sample_ids) > MAX_SAMPLES_PER_HASH
            for sample_ids in hash_counter.values()
        ),
    }
    return train, build_summary


def _validate_md5_group_stratified_folds(folded, raw_train):
    required = {
        "Id", "Answer", "No_ordering", "answer_tuple", "cluster_id", "Fold",
        "split_method", "split_seed", "split_max_samples_per_hash",
        "split_sklearn_version",
    }
    missing = required.difference(folded.columns)
    if missing:
        raise ValueError(f"Group split is missing columns: {sorted(missing)}")
    if folded["Id"].astype(str).tolist() != raw_train["Id"].astype(str).tolist():
        raise ValueError("Group split IDs/order do not match train.csv")
    if folded["Id"].duplicated().any():
        raise ValueError("Group split contains duplicate IDs")

    expected_answers = raw_train["Answer"].map(lambda value: str(ast.literal_eval(value)))
    if folded["answer_tuple"].astype(str).tolist() != expected_answers.tolist():
        raise ValueError("Group split Answer labels do not match train.csv")
    if set(folded["split_method"].astype(str)) != {GROUP_SPLIT_METHOD}:
        raise ValueError("Unexpected group split method")
    if set(folded["split_seed"].astype(int)) != {GROUP_SPLIT_SEED}:
        raise ValueError("Unexpected group split seed")
    if set(folded["split_max_samples_per_hash"].astype(int)) != {MAX_SAMPLES_PER_HASH}:
        raise ValueError("Unexpected MD5 reuse threshold")

    folded["Fold"] = folded["Fold"].astype(int)
    if set(folded["Fold"]) != set(range(GROUP_SPLIT_FOLDS)):
        raise ValueError("Expected folds 0 through 4")
    cross_fold_clusters = int(
        (folded.groupby("cluster_id")["Fold"].nunique() > 1).sum()
    )
    if cross_fold_clusters:
        raise ValueError(f"{cross_fold_clusters} MD5 clusters cross folds")

    identity_mask = folded["answer_tuple"].eq("[1, 2, 3, 4]")
    no_ordering_mask = folded["No_ordering"].astype(str).str.lower().eq("true")
    if not identity_mask.equals(no_ordering_mask):
        raise ValueError("Identity labels and No_ordering=True rows differ")
    classes_per_fold = folded.groupby("Fold")["answer_tuple"].nunique().to_dict()
    if set(classes_per_fold.values()) != {24}:
        raise ValueError(f"Every fold must contain all 24 classes: {classes_per_fold}")

    return {
        "method": GROUP_SPLIT_METHOD,
        "rows": len(folded),
        "classes": folded["answer_tuple"].nunique(),
        "identity_rows": int(identity_mask.sum()),
        "clusters": folded["cluster_id"].nunique(),
        "cross_fold_clusters": cross_fold_clusters,
        "fold_counts": {
            str(fold): int(count)
            for fold, count in folded["Fold"].value_counts().sort_index().items()
        },
        "built_with_sklearn": str(folded["split_sklearn_version"].iloc[0]),
    }


GROUP_FOLDS_CSV.parent.mkdir(parents=True, exist_ok=True)
DRIVE_GROUP_FOLDS_CSV.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_GROUP_FOLDS_CSV.is_file():
    shutil.copy2(DRIVE_GROUP_FOLDS_CSV, GROUP_FOLDS_CSV)
elif not GROUP_FOLDS_CSV.is_file():
    folded_frame, group_split_build = _build_md5_group_stratified_folds(
        TRAIN_CSV,
        IMAGE_ROOT,
    )
    temporary_csv = GROUP_FOLDS_CSV.with_suffix(".csv.tmp")
    folded_frame.to_csv(temporary_csv, index=False)
    temporary_csv.replace(GROUP_FOLDS_CSV)
    print("group split build:", json.dumps(group_split_build, indent=2))

raw_train_frame = pd.read_csv(TRAIN_CSV)
group_fold_frame = pd.read_csv(GROUP_FOLDS_CSV)
group_split_summary = _validate_md5_group_stratified_folds(
    group_fold_frame,
    raw_train_frame,
)
if not DRIVE_GROUP_FOLDS_CSV.is_file():
    temporary_drive_csv = DRIVE_GROUP_FOLDS_CSV.with_suffix(".csv.tmp")
    shutil.copy2(GROUP_FOLDS_CSV, temporary_drive_csv)
    temporary_drive_csv.replace(DRIVE_GROUP_FOLDS_CSV)
print("group split:", json.dumps(group_split_summary, indent=2))

samples = load_samples(GROUP_FOLDS_CSV, IMAGE_ROOT, require_labels=True, validate_images=True)
answer_counts = Counter(sample.rank_label for sample in samples)
fold_counts = Counter(sample.fold for sample in samples)
assert len(samples) == 9535 and len(answer_counts) == 24
assert sum(sample.no_ordering is True for sample in samples) == 1478
assert set(fold_counts) == set(range(5))
assert sum(fold_counts.values()) == 9535
print('samples/classes/folds:', len(samples), len(answer_counts), dict(sorted(fold_counts.items())))


group split: {
  "method": "md5_stratified_group_kfold_v1",
  "rows": 9535,
  "classes": 24,
  "identity_rows": 1478,
  "clusters": 8147,
  "cross_fold_clusters": 0,
  "fold_counts": {
    "0": 1915,
    "1": 1905,
    "2": 1886,
    "3": 1911,
    "4": 1918
  },
  "built_with_sklearn": "1.6.1"
}
samples/classes/folds: 9535 24 {0: 1915, 1: 1905, 2: 1886, 3: 1911, 4: 1918}


## 5. Restore resumable unpooled-token cache shards

In [12]:
import tarfile

persisted_relative_paths = set()
cache_shards = sorted(DRIVE_TOKEN_SHARDS.glob('netvlad_cache_*.tar'))
for shard in cache_shards:
    with tarfile.open(shard, 'r') as archive:
        members = [member for member in archive.getmembers() if member.isfile()]
        persisted_relative_paths.update(member.name for member in members)
        archive.extractall(LOCAL_TOKEN_CACHE, members=members, filter='data')
print('restored shards/files:', len(cache_shards), len(persisted_relative_paths))


restored shards/files: 0 0


## 6. Extract frozen ResNet-18 local descriptors and unpooled Qwen tokens

This experiment cannot reuse a ConvNeXt token cache. Each frame stores a normalized 512-D global ResNet feature for candidate grouping and 49 local 512-D descriptors for the fold-trained NetVLAD layer. Qwen image/text tokens are extracted with the same pinned model and layer selection as the token-fusion baseline.

In [13]:
import gc
import shutil

from snu_ordering.netvlad_features import ResNet18LocalFeatureExtractor
from snu_ordering.netvlad_representations import (
    cache_netvlad_representations_batched,
    find_pending_netvlad_representations,
)
from snu_ordering.qwen_representations import FrozenQwenRepresentationExtractor

pending, valid_paths = find_pending_netvlad_representations(
    samples, LOCAL_TOKEN_CACHE, qwen_layer_indices=QWEN_LAYER_INDICES
)
print('valid/pending:', len(valid_paths), len(pending))
next_shard_index = max(
    [int(path.stem.split('_')[-1]) for path in cache_shards], default=-1
) + 1
shard_buffer = []

def persist_saved(paths):
    global next_shard_index, shard_buffer
    shard_buffer.extend(paths)
    while len(shard_buffer) >= CACHE_SHARD_SIZE:
        selected = shard_buffer[:CACHE_SHARD_SIZE]
        shard_buffer = shard_buffer[CACHE_SHARD_SIZE:]
        local_tar = WORKDIR / f'netvlad_cache_{next_shard_index:04d}.tar'
        final_tar = DRIVE_TOKEN_SHARDS / local_tar.name
        temporary = final_tar.with_suffix('.tar.tmp')
        with tarfile.open(local_tar, 'w') as archive:
            for path in selected:
                archive.add(
                    path,
                    arcname=path.relative_to(LOCAL_TOKEN_CACHE).as_posix(),
                )
        shutil.copy2(local_tar, temporary)
        temporary.replace(final_tar)
        local_tar.unlink()
        print('persisted:', final_tar.name)
        next_shard_index += 1

unpersisted_valid = [
    path for path in valid_paths
    if path.relative_to(LOCAL_TOKEN_CACHE).as_posix()
    not in persisted_relative_paths
]
persist_saved(unpersisted_valid)

if pending:
    dtype = (
        torch.bfloat16
        if torch.cuda.get_device_capability(0)[0] >= 8
        else torch.float16
    )
    visual_extractor = ResNet18LocalFeatureExtractor(device='cuda')
    qwen = FrozenQwenRepresentationExtractor(device_map='auto', dtype=dtype)
    remaining = pending
    for feature_batch_size in FEATURE_BATCH_CANDIDATES:
        if not remaining:
            break
        try:
            summary = cache_netvlad_representations_batched(
                remaining,
                visual_extractor=visual_extractor,
                qwen=qwen,
                qwen_layer_indices=QWEN_LAYER_INDICES,
                batch_size=feature_batch_size,
                image_workers=IMAGE_WORKERS,
                on_batch_saved=persist_saved,
            )
            print('feature batch selected:', feature_batch_size, summary)
            remaining = []
        except torch.cuda.OutOfMemoryError:
            print('feature batch OOM; retrying smaller:', feature_batch_size)
            gc.collect()
            torch.cuda.empty_cache()
            remaining, _ = find_pending_netvlad_representations(
                samples,
                LOCAL_TOKEN_CACHE,
                qwen_layer_indices=QWEN_LAYER_INDICES,
            )
    if remaining:
        raise RuntimeError('No safe feature extraction batch was found')
    del qwen, visual_extractor
    gc.collect()
    torch.cuda.empty_cache()

if shard_buffer:
    selected, shard_buffer = shard_buffer, []
    local_tar = WORKDIR / f'netvlad_cache_{next_shard_index:04d}.tar'
    final_tar = DRIVE_TOKEN_SHARDS / local_tar.name
    with tarfile.open(local_tar, 'w') as archive:
        for path in selected:
            archive.add(
                path,
                arcname=path.relative_to(LOCAL_TOKEN_CACHE).as_posix(),
            )
    temporary = final_tar.with_suffix('.tar.tmp')
    shutil.copy2(local_tar, temporary)
    temporary.replace(final_tar)
    local_tar.unlink()

pending_after, valid_after = find_pending_netvlad_representations(
    samples, LOCAL_TOKEN_CACHE, qwen_layer_indices=QWEN_LAYER_INDICES
)
assert not pending_after and len(valid_after) == 9535


valid/pending: 0 9535
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 291MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/353 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[NetVLAD token cache] 64/9,535 | 3.83 samples/s
persisted: netvlad_cache_0000.tar
[NetVLAD token cache] 128/9,535 | 5.18 samples/s
[NetVLAD token cache] 192/9,535 | 5.95 samples/s
persisted: netvlad_cache_0001.tar
[NetVLAD token cache] 256/9,535 | 6.37 samples/s
persisted: netvlad_cache_0002.tar
[NetVLAD token cache] 320/9,535 | 6.28 samples/s
[NetVLAD token cache] 384/9,535 | 6.59 samples/s
persisted: netvlad_cache_0003.tar
[NetVLAD token cache] 448/9,535 | 6.74 samples/s
persisted: netvlad_cache_0004.tar
[NetVLAD token cache] 512/9,535 | 6.72 samples/s
[NetVLAD token cache] 576/9,535 | 6.88 samples/s
persisted: netvlad_cache_0005.tar
[NetVLAD token cache] 640/9,535 | 6.99 samples/s
persisted: netvlad_cache_0006.tar
[NetVLAD token cache] 704/9,535 | 6.94 samples/s
[NetVLAD token cache] 768/9,535 | 7.05 samples/s
persisted: netvlad_cache_0007.tar
[NetVLAD token cache] 832/9,535 | 7.06 samples/s
[NetVLAD token cache] 896/9,535 | 7.15 samples/s
persisted: netvlad_cache_0008.tar
[NetVLAD 

KeyboardInterrupt: 

## 7. Build the GPU-resident token store and compiled candidate plans

The resident loader receives the NetVLAD cache loader explicitly, so no ConvNeXt cache can be mixed into this experiment. Candidate grouping uses the frozen ResNet global vectors; the supervised NetVLAD parameters are learned independently inside each fold.

In [ ]:
import importlib
import shutil
import subprocess
import sys

# Section 7 is self-healing: the large data bundle may have restored old code.
subprocess.run(['tar', '-xf', str(CODE_BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
for cache_dir in (WORKDIR / 'snu_ordering').rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)
for name in list(sys.modules):
    if name == 'snu_ordering' or name.startswith('snu_ordering.'):
        del sys.modules[name]
sys.path.insert(0, str(WORKDIR))
importlib.invalidate_caches()

from snu_ordering.netvlad_representations import (
    load_netvlad_representation,
    netvlad_cache_path_for,
)
from snu_ordering.structured_scorer import prepare_candidate_plan
from snu_ordering.token_fusion_training import (
    TokenFusionTrainingConfig,
    prepare_resident_token_fusion_samples,
)
print('section 7 code overlay verified:', prepare_candidate_plan)

DEVICE = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')
resident_store, prepared_samples = prepare_resident_token_fusion_samples(
    samples,
    LOCAL_TOKEN_CACHE,
    DEVICE,
    top_h=TOP_H,
    storage_mode=RESIDENT_STORAGE,
    representation_loader=load_netvlad_representation,
    cache_path_builder=netvlad_cache_path_for,
)
base_config = TokenFusionTrainingConfig(
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=128,
    evaluation_batch_size=256,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    seed=SEED,
    top_h=TOP_H,
    top_m=TOP_M,
    alpha=ALPHA,
    beta=BETA,
    gamma=GAMMA,
    delta=DELTA,
    use_amp=True,
)


## 8. Probe safe train/evaluation batches for the NetVLAD scorer

In [ ]:
from dataclasses import replace

from snu_ordering.netvlad_scorer import NetVLADTokenFusionStructuredScorer
from snu_ordering.token_fusion_training import (
    probe_token_fusion_batch_size,
    probe_token_fusion_evaluation_batch_size,
)

def make_scorer(verifier=VERIFIER):
    return NetVLADTokenFusionStructuredScorer(
        qwen_layer_indices=resident_store.qwen_layer_indices,
        local_descriptor_dim=512,
        hidden_dim=HIDDEN_DIM,
        num_heads=NUM_HEADS,
        netvlad_clusters=NETVLAD_CLUSTERS,
        resampler_tokens=RESAMPLER_TOKENS,
        verifier=verifier,
        use_modality_embeddings=USE_MODALITY_EMBEDDINGS,
    ).to(DEVICE)

def plan_work(item):
    plan = item.candidate_plan
    return (
        4 * int(plan.intra_order_indices.shape[0])
        + int(plan.alignment_group_indices.shape[0]),
        plan.latent_count,
    )

probe_scorer = make_scorer()
fold0_train = [item for item in prepared_samples if item.sample.fold != 0]
worst_case_rows = sorted(fold0_train, key=plan_work, reverse=True)
micro_batch_size = probe_token_fusion_batch_size(
    probe_scorer,
    resident_store,
    worst_case_rows,
    base_config,
    candidates=(96, 64, 48, 32, 24, 16, 8),
    max_vram_fraction=0.85,
)
evaluation_batch_size = probe_token_fusion_evaluation_batch_size(
    probe_scorer,
    resident_store,
    worst_case_rows,
    base_config,
    candidates=(512, 384, 256, 192, 128, 96, 64, 48, 32),
    max_vram_fraction=0.85,
)
training_config = replace(
    base_config,
    micro_batch_size=micro_batch_size,
    evaluation_batch_size=evaluation_batch_size,
)
print(
    'selected logical/micro/eval batch:',
    training_config.batch_size,
    micro_batch_size,
    evaluation_batch_size,
)
del probe_scorer
gc.collect()
torch.cuda.empty_cache()


## 9. Run the NetVLAD five-fold experiment

Every fold uses the same cache, split, seed, logical batch, probed micro-batch, evaluation batch, optimizer, and early-stopping policy. NetVLAD assignment weights and centroids are reinitialized and trained only on that fold's training rows.

In [ ]:
import random

from snu_ordering.token_fusion_training import train_token_fusion_scorer

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

for fold in FOLDS_TO_RUN:
    output_dir = DRIVE_RESULTS_ROOT / f'fold_{fold}'
    if (output_dir / 'training_summary.json').is_file():
        print('already completed:', fold)
        continue
    train_rows = [item for item in prepared_samples if item.sample.fold != fold]
    validation_rows = [item for item in prepared_samples if item.sample.fold == fold]
    seed_everything(SEED)
    scorer = make_scorer()
    print(
        f'start NetVLAD fold={fold} '
        f'train={len(train_rows)} val={len(validation_rows)}'
    )
    train_token_fusion_scorer(
        scorer,
        resident_store,
        train_rows,
        validation_rows,
        output_dir,
        training_config,
    )
    del scorer
    gc.collect()
    torch.cuda.empty_cache()


## 10. Aggregate five-fold validation results

In [ ]:
import json

fold_records = []
for fold in range(5):
    path = DRIVE_RESULTS_ROOT / f'fold_{fold}' / 'training_summary.json'
    if not path.is_file():
        raise RuntimeError(f'missing completed fold: {path}')
    summary = json.loads(path.read_text(encoding='utf-8'))
    best = max(
        summary['history'],
        key=lambda row: (row['exact_match_accuracy'], -row['epoch']),
    )
    fold_records.append({'fold': fold, **best})

total = sum(record['samples'] for record in fold_records)
five_fold_summary = {
    'experiment': 'resnet18_netvlad_token_fusion_v1',
    'weighted_exact_match_accuracy': sum(
        record['samples'] * record['exact_match_accuracy']
        for record in fold_records
    ) / total,
    'weighted_lightweight_top8_recall': sum(
        record['samples'] * record['lightweight_top8_recall']
        for record in fold_records
    ) / total,
    'validation_samples': total,
    'folds': fold_records,
    'train_batch_size': training_config.batch_size,
    'micro_batch_size': training_config.micro_batch_size,
    'evaluation_batch_size': training_config.evaluation_batch_size,
    'resident_storage_mode': resident_store.storage_mode,
    'qwen_layer_indices': resident_store.qwen_layer_indices,
    'visual_backbone': 'resnet18_imagenet1k_v1',
    'visual_aggregator': 'netvlad',
    'netvlad_clusters': NETVLAD_CLUSTERS,
    'use_modality_embeddings': USE_MODALITY_EMBEDDINGS,
}
summary_path = DRIVE_RESULTS_ROOT / 'five_fold_summary.json'
summary_path.write_text(
    json.dumps(five_fold_summary, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(five_fold_summary, indent=2))
print('saved:', summary_path)
